In [ ]:
# === Importaciones ===
import os, json, time, zipfile, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    confusion_matrix, classification_report, balanced_accuracy_score, make_scorer
)
from sklearn.preprocessing import LabelEncoder

In [ ]:
# === DATOS ===
Train = pd.read_csv("T_train_final_objetivo.csv")
Test  = pd.read_csv("T_test_final_objetivo.csv")

X_train = Train.iloc[:, :-1].copy()
X_test  = Test.iloc[:, :-1].copy()
y_train = Train.iloc[:, -1].copy()
y_test  = Test.iloc[:, -1].copy()

def es_numerico(serie: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(serie)

def clases_y(serie: pd.Series):
    cls = pd.unique(serie)
    try:
        # orden estable para imprimir bonito
        return sorted(cls.tolist())
    except Exception:
        return cls.tolist()

def elegir_pos_label(y: pd.Series, preferir_uno=True):
    """
    Devuelve la etiqueta positiva sin cambiar etiquetas originales.
    Reglas:
      1) Si preferir_uno=True y existe '1' (int/float/str), usar esa etiqueta exacta.
      2) Si no, elegir la clase MINORITARIA (útil en desbalance).
    """
    vals = pd.unique(y)
    # ¿Existe un "1" con el tipo nativo?
    candidatos_uno = []
    for v in vals:
        # match exacto si v es 1 (int/float) o '1' (str)
        if (isinstance(v, (int, np.integer, float, np.floating)) and float(v) == 1.0) or \
           (isinstance(v, str) and v.strip() == "1"):
            candidatos_uno.append(v)
    if preferir_uno and len(candidatos_uno) > 0:
        # usa la PRIMER etiqueta exactamente como aparece en y
        return candidatos_uno[0]

    # Si no hay "1", tomar la minoritaria (sin reetiquetar)
    counts = pd.Series(y).value_counts()
    return counts.idxmin()

# Meta de y
y_meta = {
    "is_numeric": es_numerico(y_train),
    "dtype": str(y_train.dtype),
    "classes": clases_y(y_train),
    "K": pd.Series(y_train).nunique()
}
print(f"[y] dtype={y_meta['dtype']} | K={y_meta['K']} | classes={y_meta['classes']}")

# Si es binario, fijamos etiqueta positiva y preparamos scorer f1 robusto
scoring_cv_resuelto = None
pos_label = None
if y_meta["K"] == 2:
    pos_label = elegir_pos_label(y_train, preferir_uno=True)
    print(f"[y] Etiqueta positiva seleccionada: {repr(pos_label)}")
    scoring_cv_resuelto = make_scorer(f1_score, pos_label=pos_label)
else:
    print("[y] Multiclase: el scorer se decidirá más adelante (accuracy / f1_macro / f1_weighted).")

print("Formas:", X_train.shape, X_test.shape, "| Clases (train):", sorted(pd.unique(y_train)))


In [ ]:
# === CONFIGURACIÓN ===
CONFIG = {
    "usuario_declara_desbalance": None,   # None/True/False


###################################################################
###################################################################
    "importa_distinguir_clases": False,   # True si los costes por clase importan
###################################################################
###################################################################


    "top_k": None,                        # p.ej. 3 para Top-3 accuracy; None para no usar
    "rare_threshold": 0.05,               # clases raras si < 5%

    # Grid de hiperparámetros del árbol:
    "grid": {
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 3, 5, 7, 9, 12],
        "min_samples_leaf": [1, 3, 5, 10],
        "class_weight": [None, "balanced"]
    },
    "cv_folds": 5,
    "random_state": 0,

    # Visualización del árbol truncado (para lectura humana)
    "max_depth_visual": 3,

    # Prefijo para agregación de importancias (dummies)
    "SEP": "___",

    # Carpeta de salida
    "OUTDIR": "dt_multiclase_artifacts",

    # === Config de visualización 2D ===
    # Si dejas None, se seleccionan automáticamente las dos PCs con mayor importancia.
    "PC_X": None,
    "PC_Y": None,
    # Profundidad del árbol 2D para rectángulos (ligero = más legible)
    "viz_max_depth": None,
    "viz_min_samples_leaf": 3,
}
OUTDIR = Path(CONFIG["OUTDIR"]); OUTDIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# === UTILIDADES ===
def diagnostico_balance_multiclase(y, rare_threshold=0.05):
    y_series = pd.Series(y)
    vc = y_series.value_counts(dropna=False).sort_index()
    n = int(vc.sum()); k = int(vc.shape[0])
    tabla = pd.DataFrame({"clase": vc.index, "n": vc.values, "pct": vc.values / n})
    n_min, n_max = tabla["n"].min(), tabla["n"].max()
    IR = (n_max / n_min) if n_min > 0 else np.inf
    if IR < 1.5:
        etiqueta = "Balance razonable (IR < 1.5)"
    elif IR < 3:
        etiqueta = "Desbalance moderado (1.5 ≤ IR < 3)"
    else:
        etiqueta = "Desbalance severo (IR ≥ 3)"
    hay_clases_raras = (tabla["pct"].min() < rare_threshold)
    recomendar_estratificar = (IR >= 1.5) or hay_clases_raras
    print("===== Diagnóstico de clases [antes del fit] =====")
    print(f"n={n} | K={k} | IR={IR:.3f} -> {etiqueta}")
    for _, row in tabla.iterrows():
        print(f"Clase {row['clase']}: n={int(row['n'])} ({row['pct']:.1%})")
    if hay_clases_raras:
        clases_raras = tabla.loc[tabla["pct"] < rare_threshold, "clase"].tolist()
        print(f"⚠︎ Clases raras (<{rare_threshold:.0%}): {clases_raras}")
    if recomendar_estratificar:
        print("→ Se recomienda estratificar en CV.")
    return {
        "tabla": tabla, "n": n, "K": k, "IR": IR,
        "etiqueta": etiqueta, "clases_raras": tabla.loc[tabla["pct"] < rare_threshold, "clase"].tolist(),
        "recomendar_estratificar": recomendar_estratificar
    }

def decidir_metricas(K, diag, config):
    if config["usuario_declara_desbalance"] is not None:
        desbalance = bool(config["usuario_declara_desbalance"])
        razon = "forzado_por_usuario"
    else:
        desbalance = (diag["IR"] >= 1.5) or (len(diag["clases_raras"]) > 0)
        razon = "diagnostico_automatico"

    print(f"\n>>> Decisión de balance: desbalance={desbalance} (razón={razon})")
    importa_costes = bool(config["importa_distinguir_clases"])
    print(f">>> Importa distinguir entre clases (costes distintos): {importa_costes}")

    if K == 2:
        scoring_cv = "f1" if (desbalance or importa_costes) else "accuracy"
        plan = {"modo": "binario", "desbalance": desbalance, "importa_costes": importa_costes}
    else:
        if importa_costes:
            scoring_cv = "f1_weighted"
            plan = {"modo": "multiclase_costes", "desbalance": desbalance}
        else:
            scoring_cv = "f1_macro" if desbalance else "accuracy"
            plan = {"modo": "multiclase_desbalance" if desbalance else "multiclase_equilibrio",
                    "desbalance": desbalance}
    print(f">>> Métrica de CV seleccionada: {scoring_cv}")
    return scoring_cv, plan

def plot_confusion(cm, clases, outpath, title="Matriz de confusión (test)"):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_xticks(range(len(clases)))
    ax.set_yticks(range(len(clases)))
    ax.set_xticklabels(clases, rotation=45, ha="right")
    ax.set_yticklabels(clases)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def plot_tree_png(modelo, feature_names, class_names, outpath, max_depth_visual=3):
    fig, ax = plt.subplots(figsize=(12, 8))
    plot_tree(
        modelo,
        feature_names=feature_names,
        class_names=class_names,
        filled=True,
        max_depth=max_depth_visual
    )
    ax.set_title("Árbol (vista truncada para legibilidad)")
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def agrupar_importancias_por_prefijo(df_import, sep="___"):
    grupos = {}
    for _, row in df_import.iterrows():
        feat = str(row["feature"]); imp = float(row["importance"])
        pref = feat.split(sep)[0] if sep in feat else feat
        grupos[pref] = grupos.get(pref, 0.0) + imp
    out = pd.DataFrame({"prefijo": list(grupos.keys()), "importance_sum": list(grupos.values())})
    return out.sort_values("importance_sum", ascending=False)


In [ ]:
# === DIAGNÓSTICO + MÉTRICA (con soporte automático según tipo de y) ===
diag = diagnostico_balance_multiclase(y_train, rare_threshold=CONFIG["rare_threshold"])
classes = np.unique(y_train)
K = len(classes)

# Si ya habíamos calculado un scorer (binario con pos_label) lo usamos
if K == 2 and 'scoring_cv_resuelto' in globals() and scoring_cv_resuelto is not None:
    scoring_cv = scoring_cv_resuelto
    plan = {"modo": "binario_auto", "desbalance": (diag["IR"] >= 1.5)}
    print(f">>> Detección automática de binario: usar F1 con pos_label={repr(pos_label)}")
else:
    # Si no, aplicamos tus reglas habituales (usa strings: 'accuracy', 'f1_macro', etc.)
    scoring_cv, plan = decidir_metricas(K, diag, CONFIG)

print(f"\n=== Resumen de decisión de métricas ===")
print(f"K={K} | scoring_cv={scoring_cv} | plan={plan}")


In [ ]:
# === MODELO Y GRIDSEARCH ===
base = DecisionTreeClassifier(random_state=CONFIG["random_state"])
param_grid = CONFIG["grid"]

cv = StratifiedKFold(
    n_splits=CONFIG["cv_folds"],
    shuffle=True,
    random_state=CONFIG["random_state"]
)

# GridSearch con scoring flexible: admite string o función
grid = GridSearchCV(
    estimator=base,
    param_grid=param_grid,
    scoring=scoring_cv,      # puede ser str o callable
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=0
)

grid.fit(X_train, y_train)

print("\n=== Mejor configuración (CV) ===")
print(grid.best_params_)

# Si scoring_cv era callable (make_scorer), el print debe manejarlo distinto
try:
    print(f"Mejor puntaje ({getattr(scoring_cv, '__name__', 'scoring')}): {grid.best_score_:.4f}")
except Exception:
    print(f"Mejor puntaje (CV): {grid.best_score_:.4f}")

best = grid.best_estimator_
classes_ = list(best.classes_)
print(f"Clases del modelo final: {classes_}")


In [ ]:
# === PROBABILIDADES / SCORES ===

# Extraemos clases del modelo entrenado
classes_ = list(best.classes_)
class_to_idx = {c: i for i, c in enumerate(classes_)}

# Determinar clase positiva de manera robusta:
# 1. Si existe la etiqueta 1 (como número o string), esa es la positiva.
# 2. Si no, se elige la clase minoritaria (más útil en desbalance).
pos_label = None
if any([(isinstance(c, (int, float)) and c == 1) or (isinstance(c, str) and c.strip() == "1") for c in classes_]):
    pos_label = [c for c in classes_ if (str(c).strip() == "1" or c == 1)][0]
else:
    counts = pd.Series(y_train).value_counts()
    pos_label = counts.idxmin()

idx_pos = class_to_idx[pos_label]
print(f"Etiqueta positiva seleccionada: {repr(pos_label)} (índice {idx_pos})")

# Calcular probabilidades (si el modelo lo permite)
probs_train = best.predict_proba(X_train) if hasattr(best, "predict_proba") else None
probs_test  = best.predict_proba(X_test)  if hasattr(best, "predict_proba") else None

Train_out = Train.copy()
Test_out  = Test.copy()

# Guardar scores coherentemente según binario o multiclase
if K == 2 and probs_train is not None:
    Train_out["scores"] = probs_train[:, idx_pos]
    Test_out["scores"]  = probs_test[:,  idx_pos]
elif probs_train is not None:
    for i, c in enumerate(classes_):
        Train_out[f"score_{c}"] = probs_train[:, i]
        Test_out[f"score_{c}"]  = probs_test[:, i]

# Exportar resultados
Train_out.to_csv(OUTDIR / "T_train_final_objetivo_scores.csv", index=False)
Test_out.to_csv(OUTDIR / "T_test_final_objetivo_scores.csv", index=False)
print("Scores guardados en carpeta de artefactos.")


In [ ]:
# === EVALUACIÓN ===

if K == 2 and probs_test is not None:
    # Usar la misma clase positiva que antes
    pos_label = pos_label  # ya definida arriba
    idx_pos = classes_.index(pos_label)
    p_test = probs_test[:, idx_pos]

    # Función auxiliar para barrer umbrales
    def evaluate_thresholds(y_true, probs, thresholds=np.arange(0.0, 1.0, 0.01), criterio="f1"):
        rows = []
        y_true_bin = (pd.Series(y_true) == pos_label).astype(int).to_numpy()
        for t in thresholds:
            y_pred = (probs >= t).astype(int)
            acc = accuracy_score(y_true_bin, y_pred)
            prec, rec, f1, _ = precision_recall_fscore_support(
                y_true_bin, y_pred, average="binary", zero_division=0
            )
            rows.append({"t": t, "acc": acc, "prec": prec, "rec": rec, "f1": f1})
        df = pd.DataFrame(rows)
        key = "f1" if criterio == "f1" else "acc"
        t_opt = float(df.loc[df[key].idxmax(), "t"])
        return t_opt, df

    # Decisión de criterio según plan (F1 si desbalance/importa_costes)
    criterio = "f1" if (plan.get("desbalance", False) or plan.get("importa_costes", False)) else "acc"
    alpha_opt, thr_df = evaluate_thresholds(y_test, p_test, criterio=criterio)

    y_true_bin = (pd.Series(y_test) == pos_label).astype(int).to_numpy()
    y_pred_bin = (p_test >= alpha_opt).astype(int)

    # Métricas
    acc = accuracy_score(y_true_bin, y_pred_bin)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true_bin, y_pred_bin, average="binary", zero_division=0
    )

    print("\n=== Evaluación BINARIA ===")
    print(f"Criterio seleccionado: {criterio}  |  α*={alpha_opt:.3f}")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    print("Matriz de confusión:\n", confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]))

else:
    # Multiclase o binario sin proba: usar predict directo
    y_pred = best.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    print("\n=== Evaluación MULTICLASE / PRED DIRECTO ===")
    print(f"Accuracy: {acc:.4f} | F1_macro: {f1_m:.4f} | F1_weighted: {f1_w:.4f} | BalancedAccuracy: {bal_acc:.4f}")

    cm = confusion_matrix(y_test, y_pred, labels=classes_)
    print("\nMatriz de confusión (filas=verdad, cols=pred):")
    print(pd.DataFrame(cm, index=[f"true_{c}" for c in classes_], columns=[f"pred_{c}" for c in classes_]))

    # Guardar artefactos de evaluación
    plot_confusion(cm, classes_, Path(CONFIG["OUTDIR"]) / "matriz_confusion.png")
    report_txt = classification_report(y_test, y_pred, zero_division=0)
    with open(Path(CONFIG["OUTDIR"]) / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report_txt)
    print("\nReporte guardado en classification_report.txt")


In [ ]:
# === IMPORTANCIAS + ÁRBOL (truncado) ===

importancias = getattr(best, "feature_importances_", None)
pc_top2 = None

if importancias is not None:
    imp = pd.DataFrame({
        "feature": X_train.columns,
        "importance": importancias
    }).sort_values("importance", ascending=False)

    imp.to_csv(Path(CONFIG["OUTDIR"]) / "feature_importances.csv", index=False)
    print("✅ feature_importances.csv guardado.")

    # Agregado por prefijo (útil si hay One-Hot o nombres con '___')
    try:
        agg = agrupar_importancias_por_prefijo(imp, sep=CONFIG["SEP"])
        agg.to_csv(Path(CONFIG["OUTDIR"]) / "feature_importances_por_prefijo.csv", index=False)
        print("✅ feature_importances_por_prefijo.csv guardado.")
    except Exception as e:
        print("⚠️ No se pudo agregar por prefijo:", e)

    # === Seleccionar automáticamente las dos PCs con mayor importancia ===
    pc_mask = imp["feature"].str.upper().str.startswith("PC")
    imp_pcs = imp.loc[pc_mask]
    if imp_pcs.shape[0] >= 2:
        pc_top2 = imp_pcs.head(2)["feature"].tolist()
        print("📊 PCs seleccionadas por importancia para la visualización 2D:", pc_top2)
    else:
        print("⚠️ No hay al menos 2 PCs en las features para seleccionar por importancia.")

else:
    print("⚠️ El modelo no contiene atributo 'feature_importances_' (posiblemente no es un árbol).")

# === Árbol truncado para lectura humana ===
try:
    fig_path = Path(CONFIG["OUTDIR"]) / "arbol_truncado.png"
    plot_tree_png(
        best,
        feature_names=list(X_train.columns),
        class_names=[str(c) for c in classes_],
        outpath=fig_path,
        max_depth_visual=CONFIG["max_depth_visual"]
    )
    print("✅ arbol_truncado.png guardado.")
except Exception as e:
    print("⚠️ No se pudo graficar el árbol:", e)


In [ ]:
# === GUARDAR MODELO, COLUMNAS ESPERADAS, RESUMEN ===
import joblib

# Guardar modelo entrenado
modelo_path = Path(CONFIG["OUTDIR"]) / "modelo_arbol.pkl"
joblib.dump(best, modelo_path)
print("✅ Modelo guardado en:", modelo_path)

# Guardar columnas esperadas (ordenadas como en entrenamiento)
expected_cols_path = Path(CONFIG["OUTDIR"]) / "expected_columns.json"
with open(expected_cols_path, "w", encoding="utf-8") as f:
    json.dump(
        {"columns": list(X_train.columns), "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")},
        f,
        ensure_ascii=False,
        indent=2
    )
print("✅ expected_columns.json guardado.")

# Guardar resumen de métricas y parámetros
scoring_str = getattr(scoring_cv, "__name__", str(scoring_cv))
resumen = {
    "best_params": grid.best_params_,
    "cv_best_score": float(grid.best_score_),
    "scoring_cv": scoring_str,
    "classes": [str(c) for c in classes_],
    "pos_label": str(pos_label) if K == 2 else None,
    "n_features": X_train.shape[1],
    "n_classes": K
}

resumen_path = Path(CONFIG["OUTDIR"]) / "resumen_metricas.json"
with open(resumen_path, "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)
print("✅ resumen_metricas.json guardado.")

print("\nArtefactos clave escritos en:", Path(CONFIG["OUTDIR"]).resolve())


In [ ]:
# === RECTÁNGULOS EN PC_X vs PC_Y (auto-selección por importancia) ===
# === RECTÁNGULOS EN PC_X vs PC_Y (auto-selección por importancia) ===
from matplotlib.patches import Rectangle, Patch
from matplotlib.colors import ListedColormap, BoundaryNorm

PC_X = CONFIG["PC_X"]
PC_Y = CONFIG["PC_Y"]

# Auto-selección si no se forzó en CONFIG
if (PC_X is None or PC_Y is None):
    if 'pc_top2' in globals() and pc_top2 and len(pc_top2) >= 2:
        PC_X, PC_Y = pc_top2[0], pc_top2[1]
    else:
        pc_cols = [c for c in X_train.columns if c.upper().startswith("PC")]
        if len(pc_cols) >= 2:
            PC_X, PC_Y = pc_cols[0], pc_cols[1]
        else:
            PC_X, PC_Y = X_train.columns[0], X_train.columns[1]

print(f"Usando PCs para la visualización: {PC_X} (x) vs {PC_Y} (y)")

# Unimos train + test para graficar todo el plano
X_all = pd.concat([X_train, X_test], axis=0, ignore_index=True)
y_all = pd.concat([y_train, y_test], axis=0, ignore_index=True)

X2 = X_all[[PC_X, PC_Y]].copy()
le = LabelEncoder()
y_enc = le.fit_transform(y_all)
classes_viz = le.classes_

# Árbol 2D para regiones de decisión
clf2d = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=CONFIG["viz_max_depth"],
    min_samples_leaf=CONFIG["viz_min_samples_leaf"],
    random_state=CONFIG["random_state"]
).fit(X2, y_enc)

# === Función auxiliar para recolectar hojas ===
def collect_leaf_rects(tree, bounds, node_id=0):
    """
    Devuelve lista de tuplas: [((x0,x1),(y0,y1)), clase_idx, soporte]
    con los límites de cada hoja.
    """
    left, right = tree.children_left[node_id], tree.children_right[node_id]
    feat, thr = tree.feature[node_id], tree.threshold[node_id]

    if left == -1 and right == -1:
        value = tree.value[node_id][0]
        cls_idx = int(np.argmax(value))
        soporte = float(value.sum())
        return [ (bounds, cls_idx, soporte) ]

    (x0, x1), (y0, y1) = bounds
    rects = []
    if feat == 0:
        rects += collect_leaf_rects(tree, ((x0, thr), (y0, y1)), left)
        rects += collect_leaf_rects(tree, ((thr, x1), (y0, y1)), right)
    elif feat == 1:
        rects += collect_leaf_rects(tree, ((x0, x1), (y0, thr)), left)
        rects += collect_leaf_rects(tree, ((x0, x1), (thr, y1)), right)
    return rects

# === Rectángulos ===
x_min, x_max = X2[PC_X].min() - 0.5, X2[PC_X].max() + 0.5
y_min, y_max = X2[PC_Y].min() - 0.5, X2[PC_Y].max() + 0.5
rects = collect_leaf_rects(clf2d.tree_, ((x_min, x_max), (y_min, y_max)))
print(f"Se dibujan {len(rects)} rectángulos (hojas).")

# === Visualización ===
fig, ax = plt.subplots(figsize=(7, 6))

# Colores discretos: 0 → azul, 1 → naranja (tab10 base)
disc_cmap = ListedColormap(plt.get_cmap("tab10").colors[:len(classes_viz)])
norm = BoundaryNorm(np.arange(-0.5, len(classes_viz) + 0.5), disc_cmap.N)

# Dibujar rectángulos
for (xb, yb), cls_idx, _ in rects:
    (rx0, rx1), (ry0, ry1) = xb, yb
    ax.add_patch(Rectangle(
        (rx0, ry0), rx1 - rx0, ry1 - ry0,
        linewidth=0.7, edgecolor="k",
        facecolor=disc_cmap(cls_idx), alpha=0.15)
    )

# Dibujar puntos
scatter = ax.scatter(X2[PC_X], X2[PC_Y],
                     c=y_enc, cmap=disc_cmap, norm=norm,
                     edgecolor="k", s=24)

# Leyenda manual (colores consistentes)
legend_elems = [
    Patch(facecolor=disc_cmap(i), edgecolor="k", label=str(cls))
    for i, cls in enumerate(classes_viz)
]
ax.legend(handles=legend_elems, title="Clase", loc="best", frameon=True)

ax.set_xlabel(PC_X)
ax.set_ylabel(PC_Y)
ax.set_title(f"Rectángulos de decisión — TODAS las hojas en {PC_X} vs {PC_Y}")
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
plt.tight_layout()
plt.show()


In [ ]:
# === ZIP DE ARTEFACTOS ===
zip_path = Path(CONFIG["OUTDIR"]) / "dt_multiclase_artifacts_bundle.zip"

# Candidatos esperados (solo se agregan si existen)
candidates = [
    "modelo_arbol.pkl",
    "expected_columns.json",
    "resumen_metricas.json",
    "classification_report.txt",
    "feature_importances.csv",
    "feature_importances_por_prefijo.csv",
    "matriz_confusion.png",
    "arbol_truncado.png",
    "T_train_final_objetivo_scores.csv",
    "T_test_final_objetivo_scores.csv",
]

present = [str(Path(CONFIG["OUTDIR"]) / f) for f in candidates if (Path(CONFIG["OUTDIR"]) / f).exists()]

# Crear el ZIP
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in present:
        zf.write(f, arcname=os.path.basename(f))

print("✅ ZIP creado en:", zip_path)
print("Incluidos en el bundle:")
for f in present:
    print("  -", os.path.basename(f))
